In [ ]:
import numpy as np
import pandas as pd

import plotly.express as px
from pathlib import Path
import transforms as T

In [ ]:
def log_bin(column, factor = 1):
    floor = np.pow(10,np.floor(np.log10(column)))
    bin = np.floor(column/floor/factor) * floor * factor
    bin = np.where(bin == 0, floor, bin).astype(int)
    return bin

In [ ]:
root = Path("./data")
raw_path = root/"raw"
clean_path = root/"clean"
map_name = "data_map.csv"

data_map = pd.read_csv(root/map_name)

In [ ]:
df = pd.read_csv(raw_path/'pppub25.csv')
df.columns

In [ ]:
raw_df = pd.read_csv(raw_path/'pppub25.csv')
market_cols = [
    'WSAL_VAL', # Wages & Salaries
    'SEMP_VAL', # Self-Employment
    'FRSE_VAL', # Farm Income
    # 'INT_VAL',  # Interest
    # 'DIV_VAL',  # Dividends
    # 'RNT_VAL',  # Rent, Royalties, Estates, Trusts
    # 'RET_VAL'   # Pensions & Taxable Retirement Distributions
]

income_df = raw_df[market_cols]
df = raw_df[['MARSUPWT']].copy()
df['unweighted_usd'] = income_df.sum(axis = 1)
# df['unweighted_usd'] = raw_df['PTOTVAL']
df.rename({"MARSUPWT":"people"}, inplace=True, axis = 1)
df.to_csv(raw_path/'pppub25_truncated.csv', index=False)


In [ ]:
df = pd.read_csv(raw_path/'pppub25_truncated.csv')

df = df[df['unweighted_usd'] > 0]
df['people']= df['people']/100
df['usd'] = df['unweighted_usd'] * df['people']
df['bin'] = log_bin(df['unweighted_usd'], factor = 1)
df[df['bin'] < 10000] = 10000
df[df['bin'] > 1000000] = 1000000
df = df.groupby('bin').agg({'people':'sum', 'usd':'sum'})
df['people_cdf'] = df['people'].cumsum()
df['usd_cdf'] = df['usd'].cumsum()


In [ ]:
px.bar(y = df['usd'], x = df.index.astype(str))

In [ ]:
px.bar(y = df['people'], x = df.index.astype(str))

In [ ]:
from functools import reduce


files = [
    'us_population.csv',
    'employment_level.csv',
    'unemployment_level.csv',
    'not_in_labor_force.csv',
]

names = ['population', 'employment','unemployment', 'NILF']

dfs = []
for file, name in zip(files, names):
  d = pd.read_csv(clean_path / file)
  d.rename(columns={'population': name}, inplace=True)
  d.set_index('year', inplace=True)
  dfs.append(d)

df = reduce(lambda a, b: a.join(b, how='left'), dfs)

df['true_unemployment'] = df['unemployment'] + df['NILF']
df['pct_employment'] = np.floor(100* df['employment']/df['population'])
df['pct_true_unemployment'] = np.floor(100 * df['true_unemployment']/df['population'])

In [ ]:
px.line(df[['pct_employment','pct_true_unemployment']])

In [ ]:
df

In [ ]:
lines=[]

with open(raw_path/'personal_income_bea.csv', mode='r') as f:
    lines = f.readlines()

lines = lines[5:49]
lines = list(map(lambda x: x.split(',')[1:3], lines))
lines = list(map(lambda x: {'key':x[0].replace('\"','').strip(), 'value':x[1].strip()}, lines))
df = pd.DataFrame(lines)
df['value'] = pd.to_numeric(df['value'], errors='coerce')
df['value'] *= 1e9

In [ ]:

for r in df.iterrows():
    print(r)


In [ ]:
data = [
    {
        "personal_income": df.loc[df['key'] == 'Personal income', 'value'],
        "gov_aid": df.loc[df['key'] == 'Government social benefits to persons', 'value'],
        "personal_savings": df.loc[df['key'] == 'Equals: Personal saving', 'value'],
        "personal_spending": df.loc[df['key'] == 'Personal consumption expenditures', 'value']
        }
    ]

In [ ]:
data

In [ ]:
df = pd.read_csv(raw_path/'DebtPenny_19930401_20260820.csv')
df = df.sort_values(by="Record Date", ascending=True).reset_index(drop = True)
df.head()

In [ ]:
px.line(df['Total Public Debt Outstanding'])